# Evo+Swarm (GA+PSO) Sea Route Planner — Kaohsiung → Sihanoukville

This notebook computes the shortest **sea-only** route (geodesic length) between **Kaohsiung** and **Sihanoukville**, avoiding land using a land mask.  
It uses a **hybrid Evolutionary (GA) + Swarm (PSO)** approach:

- **GA (outer loop)**: explores coarse route shapes (waypoint positions)
- **PSO (inner loop)**: locally refines continuous waypoint coordinates for each GA individual

**Inputs**:
- Land mask: Natural Earth 10m land (**shapefile**, WGS84)
- Optional corridor (generated from `scgraph` maritime lines)

**Outputs**:
- GeoJSON route, CSV route
- Folium HTML map (auto-open with `webbrowser` when run locally)

## 0. Configuration

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

# 1) Land shapefile (Natural Earth 10m)
#    >>> Set to your local path <<<
LAND_SHP = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"

# 2) Start & End (Kaohsiung, Sihanoukville) — can adjust later
START_LON, START_LAT = 120.27, 22.62
END_LON,   END_LAT   = 103.52958, 10.60932  # alt: (103.497, 10.6455)

# 3) Derived AOI: pad the bbox of start/end by ±2 degrees
AOI_PADDING_DEG = 2.0

# 4) Corridor parameters
USE_SCGRAPH_CORRIDOR = True      # if scgraph import fails, will gracefully fallback
CORRIDOR_HALF_WIDTH_M = 2000.0   # buffer half-width for corridor polygon (~2 km)
CORRIDOR_PENALTY = 0.02          # lambda for small corridor "attraction" penalty (normalized)

# 5) Land safety buffer
LAND_BUFFER_M = 500.0            # 0.5 km (as requested)

# 6) Waypoints (excluding start/end)
K_WAYPOINTS = 5

# 7) Evo+Swarm parameters
RANDOM_SEED = 42

# GA parameters
GA_POP = 60
GA_GENS = 120
GA_CROSSOVER_RATE = 0.8
GA_MUTATION_RATE = 0.2
GA_ELITISM = 2

# PSO parameters (per GA individual)
PSO_PARTICLES = 24
PSO_ITERS = 40
PSO_W_START = 0.9
PSO_W_END = 0.4
PSO_C1 = 1.6
PSO_C2 = 1.6

# 8) Outputs
OUTPUT_DIR = "outputs"
HTML_NAME = "route_kaohsiung_sihanoukville.html"
GEOJSON_NAME = "route_kaohsiung_sihanoukville.geojson"
CSV_NAME = "route_kaohsiung_sihanoukville.csv"

# Ensure output dir exists
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config loaded.")

## 1. Imports & Utilities

In [ ]:
import sys, os, math, random, json, webbrowser, warnings, itertools
from typing import List, Tuple, Optional
import numpy as np

# Geo stack
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point, Polygon, box
from shapely.ops import nearest_points
from shapely import STRtree

# Projections & geodesic
from pyproj import Geod, CRS, Transformer

# Optional: geographiclib as an alternative geodesic backend
try:
    from geographiclib.geodesic import Geodesic
    GEO_LIB = Geodesic.WGS84
except Exception:
    GEO_LIB = None

# Folium map
import folium

warnings.filterwarnings("ignore")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Geodesic distance (km) between 2 lon/lat points
GEOD = Geod(ellps="WGS84")

def geodesic_km(a: Tuple[float,float], b: Tuple[float,float]) -> float:
    lon1, lat1 = a
    lon2, lat2 = b
    if GEO_LIB is not None:
        inv = GEO_LIB.Inverse(lat1, lon1, lat2, lon2)
        return inv["s12"] / 1000.0
    # fallback
    _, _, dist_m = GEOD.inv(lon1, lat1, lon2, lat2)
    return dist_m / 1000.0

def polyline_length_km(coords: List[Tuple[float,float]]) -> float:
    return sum(geodesic_km(coords[i], coords[i+1]) for i in range(len(coords)-1))

def compute_aoi(start, end, pad=2.0):
    x0 = min(start[0], end[0]) - pad
    y0 = min(start[1], end[1]) - pad
    x1 = max(start[0], end[0]) + pad
    y1 = max(start[1], end[1]) + pad
    return (x0, y0, x1, y1)

def ensure_crs4326(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    return gdf

def project_local_m(gdf: gpd.GeoDataFrame, center_lonlat: Tuple[float,float]) -> Tuple[gpd.GeoDataFrame, CRS, CRS, Transformer, Transformer]:
    """
    Project to a local metric CRS (EPSG:3857 as a simple approx for buffering small distances).
    Returns projected gdf and transformers.
    """
    crs_wgs = CRS.from_epsg(4326)
    crs_m = CRS.from_epsg(3857)  # Web Mercator (meters) — ok for small buffers
    to_m = Transformer.from_crs(crs_wgs, crs_m, always_xy=True)
    to_wgs = Transformer.from_crs(crs_m, crs_wgs, always_xy=True)
    gdf_m = gdf.to_crs(crs_m)
    return gdf_m, crs_wgs, crs_m, to_m, to_wgs

def to_linestring(coords: List[Tuple[float,float]]) -> LineString:
    return LineString(coords)

def clamp_lonlat(lon, lat):
    return max(-180, min(180, lon)), max(-90, min(90, lat))

print("Imports OK.")

## 2. Load Land Mask + Buffer (0.5 km) & Build AOI

In [ ]:
start = (START_LON, START_LAT)
end   = (END_LON,   END_LAT)

AOI = compute_aoi(start, end, AOI_PADDING_DEG)
print("AOI:", AOI)

land = gpd.read_file(LAND_SHP)
land = ensure_crs4326(land)

# Clip to AOI for speed
aoi_poly = gpd.GeoDataFrame(geometry=[box(*AOI)], crs=4326)
try:
    land = land.clip(aoi_poly)
except Exception:
    land = gpd.overlay(land, aoi_poly, how="intersection")

# Buffer in metric CRS
land_m, crs_wgs, crs_m, to_m, to_wgs = project_local_m(land, ((START_LON+END_LON)/2, (START_LAT+END_LAT)/2))
land_buf_m = land_m.buffer(LAND_BUFFER_M)
land_buf = gpd.GeoDataFrame(geometry=land_buf_m, crs=crs_m).to_crs(4326)

# Build STRtree for fast intersection checks
land_polys = [geom for geom in land_buf.geometry if geom is not None and not geom.is_empty]
land_tree = STRtree(land_polys)

print(f"Land polygons after buffer: {len(land_polys)}")

## 3. Corridor from `scgraph` maritime lines (optional)

In [ ]:
# Try importing scgraph and reconstruct maritime line segments into a corridor.
corridor_centerline = None
corridor_polygon = None

def segments_to_multiline(segs):
    ls = []
    for s in segs:
        try:
            if isinstance(s, np.ndarray) and s.shape == (2,2):
                a = (float(s[0,0]), float(s[0,1]))
                b = (float(s[1,0]), float(s[1,1]))
                ls.append(LineString([a,b]))
        except Exception:
            continue
    if not ls:
        return None
    return MultiLineString(ls)

if USE_SCGRAPH_CORRIDOR:
    try:
        # Adapted from your snippet
        import numpy as _np
        from scgraph.geographs.marnet import marnet_geograph as MNET
        import random as _random

        def _segments_from_edges_list(edges, nodes_lookup=None):
            segs = []
            for e in edges:
                try:
                    if isinstance(e, (list, tuple)) and len(e) >= 2:
                        u, v = e[0], e[1]
                        if nodes_lookup and u in nodes_lookup and v in nodes_lookup:
                            pu, pv = nodes_lookup[u], nodes_lookup[v]
                            segs.append(_np.asarray([pu, pv], dtype=float))
                    elif isinstance(e, dict):
                        if "geometry" in e and hasattr(e["geometry"], "coords"):
                            coords = _np.asarray(e["geometry"].coords, dtype=float)
                            segs.extend([coords[i:i+2] for i in range(len(coords)-1)])
                        elif "coordinates" in e and isinstance(e["coordinates"], (list, tuple)):
                            coords = _np.asarray(e["coordinates"], dtype=float)
                            if coords.ndim == 2 and coords.shape[1] == 2:
                                segs.extend([coords[i:i+2] for i in range(len(coords)-1)])
                except Exception:
                    continue
            return segs if segs else None

        def _try_graph_like_and_build_segments():
            for attr in ("graph", "_graph", "geograph", "network", "G", "_G"):
                G = getattr(MNET, attr, None)
                if G is None: 
                    continue
                # nodes
                nodes = None
                for nattr in ("nodes", "node", "vertices", "_nodes"):
                    try:
                        obj = getattr(G, nattr)
                    except Exception:
                        obj = None
                    if obj is None:
                        continue
                    try:
                        tmp = {}
                        it = obj(data=True) if callable(obj) else obj
                        for item in it:
                            if isinstance(item, tuple) and len(item) == 2:
                                nid, data = item
                                lon = data.get("longitude") or data.get("lon") or data.get("x")
                                lat = data.get("latitude")  or data.get("lat")  or data.get("y")
                                if lon is not None and lat is not None:
                                    tmp[nid] = (float(lon), float(lat))
                        if tmp:
                            nodes = tmp
                            break
                    except Exception:
                        try:
                            tmp = {}
                            for nid, data in obj.items():
                                if isinstance(data, dict):
                                    lon = data.get("longitude") or data.get("lon") or data.get("x")
                                    lat = data.get("latitude")  or data.get("lat")  or data.get("y")
                                    if lon is not None and lat is not None:
                                        tmp[nid] = (float(lon), float(lat))
                            if tmp:
                                nodes = tmp
                                break
                        except Exception:
                            pass
                # edges
                for eattr in ("edges", "_edges", "links"):
                    edges = getattr(G, eattr, None)
                    if edges is None:
                        continue
                    try:
                        edges = edges() if callable(edges) else edges
                    except Exception:
                        pass
                    segs = _segments_from_edges_list(edges, nodes_lookup=nodes)
                    if segs:
                        return segs
            return None

        def _fallback_segments_by_sampling(aoi, n_paths=40):
            from scgraph.geographs.marnet import marnet_geograph
            segs = []
            if aoi is None:
                aoi = (60, -20, 150, 40)
            x0, y0, x1, y1 = aoi
            nx, ny = 6, 5
            xs = _np.linspace(x0, x1, nx)
            ys = _np.linspace(y0, y1, ny)
            pts = [(float(x), float(y)) for x in xs for y in ys]
            for _ in range(n_paths):
                (lon1, lat1), (lon2, lat2) = _random.sample(pts, 2)
                try:
                    out = marnet_geograph.get_shortest_path(
                        origin_node={"longitude": lon1, "latitude": lat1},
                        destination_node={"longitude": lon2, "latitude": lat2},
                        output_units="km",
                    )
                    path = out.get("coordinate_path", [])
                    coords = []
                    for p in path:
                        if isinstance(p, dict):
                            lon = p.get("longitude") or p.get("lon") or p.get("x")
                            lat = p.get("latitude")  or p.get("lat")  or p.get("y")
                            if lon is not None and lat is not None:
                                coords.append((float(lon), float(lat)))
                        elif isinstance(p, (list, tuple)) and len(p) >= 2:
                            a, b = p[0], p[1]
                            if -180 <= a <= 180 and -90 <= b <= 90:
                                coords.append((float(a), float(b)))
                            elif -90 <= a <= 90 and -180 <= b <= 180:
                                coords.append((float(b), float(a)))
                    if len(coords) >= 2:
                        for i in range(len(coords)-1):
                            segs.append(_np.asarray([coords[i], coords[i+1]], dtype=float))
                except Exception:
                    continue
            return segs

        segs = _try_graph_like_and_build_segments()
        if not segs:
            segs = _fallback_segments_by_sampling(AOI)

        # Filter by AOI
        def in_aoi(pt, aoi):
            x, y = pt
            x0, y0, x1, y1 = aoi
            return (x0 <= x <= x1) and (y0 <= y <= y1)

        segs2 = []
        for s in segs:
            if s is None or len(s) != 2: continue
            if in_aoi(s[0], AOI) or in_aoi(s[1], AOI):
                segs2.append(s)

        ml = segments_to_multiline(segs2)
        if ml is not None:
            # Build corridor polygon by buffering in meters
            gdf_ml = gpd.GeoDataFrame(geometry=[ml], crs=4326)
            gdf_ml_m = gdf_ml.to_crs(3857)
            corridor_polygon_m = gdf_ml_m.buffer(CORRIDOR_HALF_WIDTH_M)
            corridor_polygon = gpd.GeoDataFrame(geometry=corridor_polygon_m, crs=3857).to_crs(4326).geometry.unary_union
            corridor_centerline = ml  # keep as centerline for small penalty/projection
            print("Corridor constructed from scgraph lines.")
        else:
            print("No maritime lines found for corridor; proceeding without corridor.")
            corridor_centerline = None
            corridor_polygon = None

    except Exception as e:
        print("scgraph import or parsing failed; proceeding without corridor. Error:", e)
        corridor_centerline = None
        corridor_polygon = None
else:
    print("Corridor generation disabled by config.")

## 4. Feasibility, Repair & Fitness

In [ ]:
# --- Land intersection (segment) ---
def segment_crosses_land(p0, p1) -> bool:
    line = LineString([p0, p1])
    # quick bbox candidates
    candidates = land_tree.query(line)
    for poly in candidates:
        if line.intersects(poly):
            return True
    return False

def path_crosses_land(coords: List[Tuple[float,float]]) -> bool:
    for i in range(len(coords)-1):
        if segment_crosses_land(coords[i], coords[i+1]):
            return True
    return False

# --- Simple sea projection (if a point lies on land, push to nearest sea) ---
def project_point_to_sea(pt: Tuple[float,float]) -> Tuple[float,float]:
    p = Point(pt)
    # If not inside any land buffer polygon, return as is
    candidates = land_tree.query(p)
    for poly in candidates:
        if p.within(poly):
            # move to nearest boundary point
            np1, np2 = nearest_points(p, poly.exterior)
            return (float(np2.x), float(np2.y))
    return pt

# --- Corridor projection & distance ---
def project_point_to_corridor(pt: Tuple[float,float]) -> Tuple[float,float]:
    if corridor_polygon is None:
        return pt
    p = Point(pt)
    if corridor_polygon.contains(p):
        return pt
    # project to nearest boundary if outside
    np1, np2 = nearest_points(p, corridor_polygon)
    return (float(np2.x), float(np2.y))

def corridor_offset_metric(coords: List[Tuple[float,float]]) -> float:
    if corridor_centerline is None:
        return 0.0
    # sample every vertex; compute mean distance (km) to centerline
    dists_km = []
    for c in coords:
        p = Point(c)
        # distance in degrees; convert via local scale by geodesic to a nearby projected point
        # We'll compute nearest point along centerline, then geodesic_km
        try:
            np1, np2 = nearest_points(p, corridor_centerline)
            dists_km.append(geodesic_km((p.x, p.y), (np2.x, np2.y)))
        except Exception:
            dists_km.append(0.0)
    if not dists_km:
        return 0.0
    # Normalize by total length to keep penalty small compared to length
    length_km = polyline_length_km(coords)
    if length_km <= 1e-6:
        return 0.0
    return (sum(dists_km) / len(dists_km)) / max(1.0, length_km)

# --- Fitness ---
BIG_PENALTY_KM = 1000.0  # add if land cross

def fitness(coords: List[Tuple[float,float]]) -> float:
    L = polyline_length_km(coords)
    land_cross = path_crosses_land(coords)
    Dcorr = corridor_offset_metric(coords) if corridor_centerline is not None else 0.0
    score = -L - (CORRIDOR_PENALTY * Dcorr)
    if land_cross:
        score -= BIG_PENALTY_KM
    return score

## 5. Initialization (great-circle skeleton + small perturbations + sea & corridor repair)

In [ ]:
def linspace_route(a, b, k):
    """Linear (lon/lat) interpolation to get k waypoints between a and b (coarse)."""
    xs = np.linspace(a[0], b[0], k+2)[1:-1]
    ys = np.linspace(a[1], b[1], k+2)[1:-1]
    return list(zip(xs, ys))

def random_perturb(pt, lon_scale=0.2, lat_scale=0.2):
    """Add small random delta in degrees (tuned for AOI)."""
    return (pt[0] + np.random.normal(0, lon_scale),
            pt[1] + np.random.normal(0, lat_scale))

def init_individual(k: int) -> List[Tuple[float,float]]:
    # base skeleton
    mids = linspace_route(start, end, k)
    # apply small perturbations (scaled by AOI size)
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0) * 0.01   # 1% of width
    lat_scale = (y1-y0) * 0.01   # 1% of height
    mids = [random_perturb(pt, lon_scale, lat_scale) for pt in mids]

    # optionally snap one random waypoint to corridor
    if corridor_centerline is not None:
        j = np.random.randint(0, len(mids))
        mids[j] = project_point_to_corridor(mids[j])

    # ensure points at sea
    mids = [project_point_to_sea(pt) for pt in mids]

    # assemble full route
    coords = [start] + mids + [end]

    # if still infeasible, try a quick repair pass: move each interior point slightly toward sea/corridor
    if path_crosses_land(coords):
        mids2 = []
        for pt in mids:
            pt = project_point_to_sea(pt)
            if corridor_centerline is not None:
                pt = project_point_to_corridor(pt)
            mids2.append(pt)
        coords = [start] + mids2 + [end]

    return coords

print("Init helpers ready.")

## 6. GA (outer) + PSO (inner)

In [ ]:
# --- PSO local refine for a given route (only optimizes interior waypoints) ---
def pso_refine(coords: List[Tuple[float,float]], iters=PSO_ITERS, particles=PSO_PARTICLES):
    # Flatten interior waypoints to vector x = [lon1,lat1, ..., lonk,latk]
    interior = coords[1:-1]
    x0 = np.array([v for pt in interior for v in pt], dtype=float)

    # bounds from AOI
    x_min = []
    x_max = []
    for i in range(len(x0)//2):
        x_min += [AOI[0], AOI[1]]
        x_max += [AOI[2], AOI[3]]
    x_min = np.array(x_min); x_max = np.array(x_max)

    def vec_to_coords(vec):
        mids = [(vec[2*i], vec[2*i+1]) for i in range(len(vec)//2)]
        return [coords[0]] + mids + [coords[-1]]

    def clamp_vec(vec):
        return np.clip(vec, x_min, x_max)

    # Initialize swarm
    rng = np.random.default_rng(RANDOM_SEED)
    swarm_x = np.tile(x0, (particles,1)) + rng.normal(0, 0.01*(x_max-x_min), size=(particles, len(x0)))
    swarm_v = rng.normal(0, 0.005*(x_max-x_min), size=(particles, len(x0)))

    pbest_x = swarm_x.copy()
    pbest_f = np.array([fitness(vec_to_coords(xx)) for xx in pbest_x])

    # lbest ring topology
    idxs = np.arange(particles)
    left = (idxs - 1) % particles
    right = (idxs + 1) % particles

    def inertia_weight(t):
        return PSO_W_START + (PSO_W_END - PSO_W_START) * (t / max(1, iters-1))

    for t in range(iters):
        # compute local best per particle
        lbest = []
        for i in range(particles):
            neighbors = [i, left[i], right[i]]
            j = neighbors[np.argmax(pbest_f[neighbors])]
            lbest.append(pbest_x[j])
        lbest = np.array(lbest)

        w = inertia_weight(t)
        r1 = rng.random(size=swarm_x.shape)
        r2 = rng.random(size=swarm_x.shape)

        swarm_v = (w*swarm_v
                   + PSO_C1*r1*(pbest_x - swarm_x)
                   + PSO_C2*r2*(lbest - swarm_x))
        swarm_x = clamp_vec(swarm_x + swarm_v)

        # Repair: project any interior point that fell on land/corridor issues
        for i in range(particles):
            mids = [(swarm_x[i,2*j], swarm_x[i,2*j+1]) for j in range(len(x0)//2)]
            mids2 = []
            for pt in mids:
                pt = project_point_to_sea(pt)
                if corridor_centerline is not None:
                    pt = project_point_to_corridor(pt)
                mids2.append(pt)
            # write back
            for j,pt in enumerate(mids2):
                swarm_x[i,2*j] = pt[0]
                swarm_x[i,2*j+1] = pt[1]

        # evaluate
        fvals = np.array([fitness(vec_to_coords(xx)) for xx in swarm_x])

        # update pbest
        improved = fvals > pbest_f
        pbest_x[improved] = swarm_x[improved]
        pbest_f[improved] = fvals[improved]

    # return best
    j = int(np.argmax(pbest_f))
    return vec_to_coords(pbest_x[j])

# --- GA outer loop ---
def tournament_select(pop, fits, k=2):
    idxs = np.random.choice(len(pop), size=k, replace=False)
    best = idxs[0]
    for i in idxs[1:]:
        if fits[i] > fits[best]:
            best = i
    return pop[best]

def crossover(parent1, parent2):
    # two strategies: splice OR blend (choose randomly)
    r = np.random.rand()
    p1_mid = parent1[1:-1]; p2_mid = parent2[1:-1]
    if r < 0.5:
        # splice at a random cut
        cut = np.random.randint(1, len(p1_mid))
        child_mid = p1_mid[:cut] + p2_mid[cut:]
    else:
        # blend each waypoint (BLX-like)
        child_mid = []
        for (a,b) in zip(p1_mid, p2_mid):
            alpha = np.random.uniform(-0.2, 1.2)  # small extrapolation
            cx = a[0] + alpha*(b[0]-a[0])
            cy = a[1] + alpha*(b[1]-a[1])
            child_mid.append((cx, cy))
    # repair to sea/corridor
    child_mid = [project_point_to_sea(pt) for pt in child_mid]
    if corridor_centerline is not None:
        child_mid = [project_point_to_corridor(pt) for pt in child_mid]
    return [parent1[0]] + child_mid + [parent1[-1]]

def mutate(ind, rate=GA_MUTATION_RATE):
    mids = ind[1:-1]
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0) * 0.01
    lat_scale = (y1-y0) * 0.01
    mids2 = []
    for pt in mids:
        if np.random.rand() < rate:
            pt = (pt[0] + np.random.normal(0, lon_scale),
                  pt[1] + np.random.normal(0, lat_scale))
        pt = project_point_to_sea(pt)
        if corridor_centerline is not None:
            pt = project_point_to_corridor(pt)
        mids2.append(pt)
    return [ind[0]] + mids2 + [ind[-1]]

def evo_swarm_optimize():
    # init population
    pop = [init_individual(K_WAYPOINTS) for _ in range(GA_POP)]
    fits = np.array([fitness(ind) for ind in pop])

    best_route = pop[int(np.argmax(fits))]
    best_fit = float(np.max(fits))

    history = [(-best_fit, best_fit)]  # store (distance_km, fitness)

    for g in range(GA_GENS):
        new_pop = []

        # elitism
        elite_idx = np.argsort(-fits)[:GA_ELITISM]
        for i in elite_idx:
            new_pop.append(pop[i])

        # fill rest
        while len(new_pop) < GA_POP:
            p1 = tournament_select(pop, fits, k=3)
            p2 = tournament_select(pop, fits, k=3)

            child = p1
            if np.random.rand() < GA_CROSSOVER_RATE:
                child = crossover(p1, p2)
            child = mutate(child, rate=GA_MUTATION_RATE)

            # Local PSO refine
            child = pso_refine(child, iters=PSO_ITERS, particles=PSO_PARTICLES)

            new_pop.append(child)

        pop = new_pop
        fits = np.array([fitness(ind) for ind in pop])

        # track best
        idx = int(np.argmax(fits))
        if fits[idx] > best_fit:
            best_fit = float(fits[idx])
            best_route = pop[idx]

        if (g+1) % 10 == 0 or g == 0:
            dist_km = polyline_length_km(best_route)
            print(f"Gen {g+1:3d} | best distance: {dist_km:.2f} km")
            history.append((dist_km, best_fit))

    return best_route, history

## 7. Run Optimization

In [ ]:
best_route, hist = evo_swarm_optimize()
best_dist = polyline_length_km(best_route)
print(f"\nBest distance = {best_dist:.2f} km")

# Save GeoJSON & CSV
geo = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "Kaohsiung→Sihanoukville", "distance_km": best_dist},
        "geometry": {
            "type": "LineString",
            "coordinates": [[lon,lat] for (lon,lat) in best_route]
        }
    }]
}
geo_path = os.path.join(OUTPUT_DIR, GEOJSON_NAME)
with open(geo_path, "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)

csv_path = os.path.join(OUTPUT_DIR, CSV_NAME)
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("lon,lat\n")
    for lon,lat in best_route:
        f.write(f"{lon},{lat}\n")

print("Saved:", geo_path)
print("Saved:", csv_path)

## 8. Folium Map & Open in Browser

In [ ]:
m = folium.Map(location=[(START_LAT+END_LAT)/2, (START_LON+END_LON)/2], zoom_start=5, tiles="cartodbpositron")

# Add land polygons (simplified for web map)
try:
    land_simplified = land.simplify(0.01, preserve_topology=True)
    folium.GeoJson(land_simplified.__geo_interface__, name="Land", style_function=lambda x: {
        "fillColor": "#e6e6e6", "color": "#333333", "weight": 0.5, "fillOpacity": 1.0
    }).add_to(m)
except Exception:
    pass

# Add corridor if exists
if corridor_polygon is not None:
    try:
        folium.GeoJson(corridor_polygon.__geo_interface__, name="Corridor", style_function=lambda x: {
            "fillColor": "#93c5fd", "color": "#60a5fa", "weight": 0.5, "fillOpacity": 0.3
        }).add_to(m)
    except Exception:
        pass

# Add route
folium.PolyLine([(lat, lon) for (lon,lat) in best_route], color="#1f2937", weight=4, opacity=0.9, tooltip=f"Best route: {best_dist:.2f} km").add_to(m)

# Markers
folium.Marker(location=[START_LAT, START_LON], popup="Start: Kaohsiung").add_to(m)
folium.Marker(location=[END_LAT, END_LON], popup="End: Sihanoukville").add_to(m)

html_path = os.path.join(OUTPUT_DIR, HTML_NAME)
m.save(html_path)
print("Saved map:", html_path)

# Auto-open (works on local VSCode environment)
abs_html = os.path.abspath(html_path)
try:
    webbrowser.open(f"file:///{abs_html}")
except Exception as e:
    print("Open in browser failed, please open manually:", abs_html)